# Memory Maze: Interactive Environment Tour

Explore the Memory Maze benchmark step-by-step. Pick a backend, reset, and execute
individual actions to see first-person view, top-down map, and debug observations.

## Three backends

| | MuJoCo | Genesis Rasterizer | Genesis BatchRenderer |
|---|---|---|---|
| Top-down camera | yes | yes | no |
| ExtraObs (dict) | yes | yes | no |
| Oracle overlay | yes | yes | no |
| GPU batched physics | no | no | yes |
| Use case | reference / debug | debug / single-env | training |

**Time**: ~2 minutes per config

## 1. Configuration

Change `ENGINE` and `MODE` below, then **Restart & Run All** (or run cells one by one).

- `ENGINE="mujoco"` — original MuJoCo backend, all features
- `ENGINE="genesis"` + `MODE="rasterizer"` — single-env Genesis, all features
- `ENGINE="genesis"` + `MODE="batched"` — BatchRenderer, first-person only, fastest

In [ ]:
ENGINE = "genesis"     # "mujoco" or "genesis"
MODE = "rasterizer"    # "rasterizer" or "batched" (ignored if ENGINE="mujoco")
MAZE_SIZE = 9          # 9, 11, 13, 15

## 2. Initialization

In [ ]:
%matplotlib inline

import sys, os

import pathlib
PROJECT_ROOT = str(pathlib.Path.cwd().parent) if pathlib.Path('../train_impala.py').exists() else str(pathlib.Path.cwd())
sys.path.insert(0, PROJECT_ROOT)
if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
else:
    os.environ.setdefault("MUJOCO_GL", "glfw")

if os.path.exists('/workspace'):
    os.makedirs('/workspace/taichi_cache', exist_ok=True)
    os.environ.setdefault('TI_CACHE_PATH', '/workspace/taichi_cache')
os.environ.setdefault('GENESIS_SKIP_TK_INIT', '1')

import gym
import numpy as np
import matplotlib.pyplot as plt

# Validate config
assert ENGINE in ("mujoco", "genesis"), f"Unknown ENGINE: {ENGINE!r}"
assert MODE in ("rasterizer", "batched"), f"Unknown MODE: {MODE!r}"
size_key = f"{MAZE_SIZE}x{MAZE_SIZE}"

BATCHED = (ENGINE == "genesis" and MODE == "batched")

print(f"Engine: {ENGINE}, Mode: {MODE}, Maze: {size_key}, Batched: {BATCHED}")

## 3. Create environment

- **MuJoCo**: two CPU envs (ExtraObs egocentric + Top-down) — no GPU memory concern
- **Genesis rasterizer**: one scene with two cameras (`ExtraObs-Top-Genesis`); obs image = top-down, first-person rendered manually from `env._scene`
- **Batched**: single env, first-person only

In [ ]:
env = None       # main env (always exists)
env_top = None   # separate top-down env (MuJoCo only)
HAS_TOP = False
HAS_EXTRA = False
# For Genesis: obs['image'] is top-down; first-person comes from env._scene
_genesis_top_in_obs = False

if BATCHED:
    import genesis as gs
    if not gs._initialized:
        print('Initializing Genesis (CUDA)... may take 30-90s on first run')
        gs.init(backend=gs.cuda, logging_level='warning')
        print('Done')

    from memory_maze.genesis_backend import BatchGenesisMemoryMazeEnv
    _batch_env = BatchGenesisMemoryMazeEnv(n_envs=1, maze_size=MAZE_SIZE)

    class _SingleBatchWrapper:
        """Adapts BatchGenesisMemoryMazeEnv(n_envs=1) to single-env interface."""
        def __init__(self, batch_env):
            self._env = batch_env
            self.action_space = batch_env.action_space
            self.observation_space = batch_env.observation_space
        def reset(self):
            return self._env.reset()[0]
        def step(self, action):
            obs, rew, done, info = self._env.step([action])
            return obs[0], rew[0], done[0], info[0] if isinstance(info, list) else info
        def close(self):
            self._env.close()

    env = _SingleBatchWrapper(_batch_env)
    print(f"BatchRenderer: obs={env.observation_space.shape}, actions={env.action_space.n}")

elif ENGINE == "genesis":
    import genesis as gs
    if not gs._initialized:
        print('Initializing Genesis... may take 30-90s on first run')
        gs.init(logging_level='warning')
        print('Done')

    # One scene, two cameras. obs['image'] = top-down, egocentric via _scene.
    # use_batch_renderer=False: force OpenGL Rasterizer even when CUDA + gs_madrona
    # are available (BatchRenderer needs Vulkan which may not work in containers).
    env_id = f"memory_maze:MemoryMaze-{size_key}-ExtraObs-Top-Genesis-v0"
    env = gym.make(env_id, disable_env_checker=True, use_batch_renderer=False)
    HAS_TOP = True
    HAS_EXTRA = True
    _genesis_top_in_obs = True
    print(f"Genesis ({env_id}): 1 scene, 2 cameras")

else:
    # MuJoCo: two CPU envs — ExtraObs (egocentric) + Top (top-down)
    extra_id = f"memory_maze:MemoryMaze-{size_key}-ExtraObs-v0"
    top_id = f"memory_maze:MemoryMaze-{size_key}-Top-v0"
    env = gym.make(extra_id, disable_env_checker=True)
    env_top = gym.make(top_id, disable_env_checker=True)
    HAS_TOP = True
    HAS_EXTRA = True
    print(f"MuJoCo: {extra_id} + {top_id}")

step_count = 0
print(f"Top-down: {HAS_TOP}, ExtraObs: {HAS_EXTRA}")
print("Ready.")

## 4. Display helper

`show_step()` renders 1-3 panels depending on what's available:
1. **First-person view** (always)
2. **Top-down view** (non-batched only)
3. **ExtraObs** text (non-batched only)

In [ ]:
ACTION_NAMES = {
    0: "Noop",
    1: "Forward",
    2: "Turn Left",
    3: "Turn Right",
    4: "Forward+Left",
    5: "Forward+Right",
}

def _extract_views(obs, obs_top_separate=None):
    """Extract first-person image, top-down image, and ExtraObs dict from obs.

    Returns (img_fp, img_top_or_None, extra_dict_or_None).
    """
    if isinstance(obs, dict):
        extra = {k: v for k, v in obs.items() if k != 'image'}
        if _genesis_top_in_obs:
            # Genesis ExtraObs-Top: obs['image'] is top-down, egocentric from scene
            img_top = obs['image']
            img_fp = env.unwrapped._scene.render_egocentric()
        else:
            # MuJoCo ExtraObs: obs['image'] is egocentric
            img_fp = obs['image']
            img_top = obs_top_separate
    else:
        # Batched: raw array, no extras
        img_fp = obs
        img_top = None
        extra = None
    return img_fp, img_top, extra


def show_step(obs, obs_top_separate=None, action_name="reset", reward=0.0, done=False):
    """Display first-person, optional top-down, and ExtraObs."""
    img_fp, img_top, extra = _extract_views(obs, obs_top_separate)

    n_panels = 1 + (img_top is not None)
    fig, axes = plt.subplots(1, n_panels, figsize=(4 * n_panels, 4))
    if n_panels == 1:
        axes = [axes]

    axes[0].imshow(img_fp)
    axes[0].set_title(f"First-Person ({img_fp.shape[0]}x{img_fp.shape[1]})")
    axes[0].axis("off")

    if img_top is not None:
        axes[1].imshow(img_top)
        axes[1].set_title(f"Top-Down ({img_top.shape[0]}x{img_top.shape[1]})")
        axes[1].axis("off")

    status = f"step {step_count} | {action_name} | r={reward:.1f}"
    if done:
        status += " | DONE"
    fig.suptitle(status, fontsize=12)
    plt.tight_layout()
    plt.show()

    if extra:
        print("ExtraObs:")
        for key, val in sorted(extra.items()):
            if isinstance(val, np.ndarray):
                if val.size <= 6:
                    print(f"  {key:35s} = {val}")
                else:
                    print(f"  {key:35s}  shape={str(val.shape):15s}  dtype={val.dtype}")
            else:
                print(f"  {key:35s} = {val}")

In [ ]:
def do_step(action):
    """Step all envs with the same action and display."""
    global step_count
    obs, reward, done, info = env.step(action)
    obs_top_sep = None
    if env_top is not None:
        obs_top_sep, _, _, _ = env_top.step(action)
    step_count += 1
    show_step(obs, obs_top_sep, action_name=ACTION_NAMES[action], reward=reward, done=done)
    return obs, reward, done

## 5. Reset

Reset all envs and display the initial observation.

In [ ]:
step_count = 0
obs = env.reset()
obs_top_sep = env_top.reset() if env_top is not None else None
show_step(obs, obs_top_sep, action_name="reset")

## 6. Movement

Run these cells one by one in any order. Each executes one action.

Actions: 0=Noop, 1=Forward, 2=Turn Left, 3=Turn Right, 4=Forward+Left, 5=Forward+Right

In [ ]:
# Forward
do_step(1)

In [ ]:
# Turn Left
do_step(2)

In [ ]:
# Turn Right
do_step(3)

In [ ]:
# Forward + Left
do_step(4)

In [ ]:
# Forward + Right
do_step(5)

In [ ]:
# Noop
do_step(0)

In [ ]:
# Run N random steps
N = 20
total_reward = 0.0
for i in range(N):
    action = env.action_space.sample()
    obs, reward, done, info = env.step(action)
    if env_top is not None:
        obs_top_sep, _, _, _ = env_top.step(action)
    step_count += 1
    total_reward += reward
    if done:
        print(f"Episode ended at step {step_count}")
        break

obs_top_final = obs_top_sep if env_top is not None else None
show_step(obs, obs_top_final, action_name=f"{N} random", reward=total_reward)
print(f"Total reward over {N} random steps: {total_reward:.1f}")

## 7. Cleanup

In [ ]:
env.close()
if env_top is not None:
    env_top.close()
print("Done!")